In [1]:
import pandas as pd
df = pd.read_csv("cholera_jhu_dataset.csv")

In [2]:
#check duplicates among Location + TL + TR
duplicate_mask = df.duplicated(subset=['Location','TL','TR'], keep='first')

duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR):", duplicate_count)

Total records: 232175
Duplicate records (same Location + TL + TR): 144033


In [4]:
# Count how many records are real vs ghost
real_count = df['processing_notes'].str.contains("Phantom: False").sum()
ghost_count = df['processing_notes'].str.contains("Phantom: True").sum()

print("Real records:", real_count)
print("Ghost records:", ghost_count)

Real records: 232095
Ghost records: 80


In [3]:
# Mark duplicates (extra copies beyond the first) with 1
df['delete_flag'] = 0
duplicate_mask = df.duplicated(subset=['Location','TL','TR'], keep='first')
df.loc[duplicate_mask, 'delete_flag'] = 1

# Check counts
print("Duplicates marked for deletion:", df['delete_flag'].sum())
print("Remaining real records:", len(df) - df['delete_flag'].sum())

Duplicates marked for deletion: 144033
Remaining real records: 88142


In [5]:
# Mark ghost records (phantom = True) with 1
df.loc[df['processing_notes'].str.contains("Phantom: True"), 'delete_flag'] = 1

In [6]:
# Check counts
print("Duplicates marked for deletion:", df['delete_flag'].sum())
print("Remaining real records:", len(df) - df['delete_flag'].sum())

Duplicates marked for deletion: 144053
Remaining real records: 88122


In [7]:
# Filter to keep only real records (delete_flag = 0)
df_real = df[df['delete_flag'] == 0]

# Save to a new CSV file
df_real.to_csv("real_non_phantom_cholera_records.csv", index=False)

print("Real non phantom records saved")

Real non phantom records saved


In [21]:
df = pd.read_csv("real_non_phantom_cholera_records.csv")
# Count how many "::" separators each Location has
df['location_parts'] = df['Location'].str.count("::")

counts = df['location_parts'].value_counts().sort_index()

print("Records by Location format:")
for depth, count in counts.items():
    if depth == 1:
        print(f"{count} records at Country level (AFR::CMR)")
    elif depth == 2:
        print(f"{count} records at Region level (AFR::CMR::South-West)")
    elif depth == 3:
        print(f"{count} records at District level (AFR::CMR::South-West::Nguti Health District)")
    else:
        print(f"{count} records with {depth} parts")

Records by Location format:
641 records at Country level (AFR::CMR)
1142 records at Region level (AFR::CMR::South-West)
86190 records at District level (AFR::CMR::South-West::Nguti Health District)
128 records with 4 parts
21 records with 5 parts


In [22]:
# Clean spaces around ::
df["Location_clean"] = (
    df["Location"]
    .astype(str)
    .str.replace(r"\s*::\s*", "::", regex=True)
    .str.strip()
)

# Keep only first 4 location components
def standardize_location(location):
    parts = location.split("::")
    return "::".join(parts[:4]) if len(parts) > 4 else location

df["Location_std"] = df["Location_clean"].apply(standardize_location)

In [23]:
df[df["Location_clean"].str.contains("Mayo-Louti", na=False)][
    ["Location", "Location_clean", "Location_std"]
].drop_duplicates()

,Location,Location_clean,Location_std
85724,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza,AFR::CMR::North::Mayo-Louti
85725,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo,AFR::CMR::North::Mayo-Louti
85728,AFR::CMR::North::Mayo-Louti::Guider,AFR::CMR::North::Mayo-Louti::Guider,AFR::CMR::North::Mayo-Louti
85865,AFR::CMR::North::Mayo-Louti::Figuil,AFR::CMR::North::Mayo-Louti::Figuil,AFR::CMR::North::Mayo-Louti
85868,AFR::CMR::North::Mayo-Louti::Guider::Golombe,AFR::CMR::North::Mayo-Louti::Guider::Golombe,AFR::CMR::North::Mayo-Louti
85871,AFR::CMR::North::Mayo-Louti::Mayo-Oulo,AFR::CMR::North::Mayo-Louti::Mayo-Oulo,AFR::CMR::North::Mayo-Louti


In [24]:
changed = (df["Location_clean"] != df["Location_std"]).sum()

print(f"{changed} records were standardized.")

149 records were standardized.


In [25]:
df["Location"] = df["Location_std"]

df = df.drop(columns=["Location_clean", "Location_std", "std_parts"], errors="ignore")

df.to_csv("standardized_non_phantom__cholera_records.csv", index=False)

In [26]:
df = pd.read_csv("standardized_non_phantom__cholera_records.csv")
duplicate_mask = df.duplicated(subset=['Location','TL','TR','sCh','cCh'], keep='first')

# Count how many duplicates exist
duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR + sCH + Cch):", duplicate_count)

Total records: 88122
Duplicate records (same Location + TL + TR + sCH + Cch): 3


In [28]:
import pandas as pd

df = pd.read_csv("standardized_non_phantom__cholera_records.csv")

df = df.drop_duplicates(
    subset=['Location','TL','TR','sCh','cCh'],
    keep='first'
)
df.to_csv(
    "standardized_cholera_records_no_duplicates.csv",
    index=False
)

print("Remaining records:", len(df))

Remaining records: 88119


In [30]:
df = pd.read_csv("standardized_non_phantom__cholera_records.csv")
#check duplicates among Location + TL + TR +sch,cch
duplicate_mask = df.duplicated(subset=['Location','TL','TR','sCh','cCh'], keep='first')

duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR ):", duplicate_count)

Total records: 88122
Duplicate records (same Location + TL + TR ): 3


In [31]:
df.to_csv("Master_non_phantom_Dataset.csv")


In [32]:
df["Level"] = df["Location"].str.count("::")
df["Level"].unique()

array([3, 1, 2])

In [33]:
df = pd.read_csv("Master_non_phantom_Dataset.csv")

df['location_parts'] = df['Location'].str.count("::")

counts = df['location_parts'].value_counts().sort_index()

print("Records by Location format:")
for depth, count in counts.items():
    if depth == 1:
        print(f"{count} records at Country level (AFR::CMR)")
    elif depth == 2:
        print(f"{count} records at Region level (AFR::CMR::South-West)")
    elif depth == 3:
        print(f"{count} records at District level (AFR::CMR::South-West::Nguti Health District)")
    else:
        print(f"{count} records with {depth} parts")

Records by Location format:
641 records at Country level (AFR::CMR)
1142 records at Region level (AFR::CMR::South-West)
86339 records at District level (AFR::CMR::South-West::Nguti Health District)


In [34]:
import pandas as pd

df = pd.read_csv("Master_non_phantom_Dataset.csv")

df["location_parts"] = df["Location"].astype(str).str.count("::")

# Check counts
print(df["location_parts"].value_counts().sort_index())

# Extract district-level records
district_df = df[df["location_parts"] == 3].copy()

print("District-level records:", len(district_df))

district_df.to_csv("district_non_phantom_dataset.csv", index=False)

print("Saved successfully.")

location_parts
1      641
2     1142
3    86339
Name: count, dtype: int64
District-level records: 86339
Saved successfully.


In [35]:
import pandas as pd

df = pd.read_csv("district_non_phantom_dataset.csv")

# Create Region column
df["Region"] = df["Location"].apply(
    lambda x: str(x).split("::")[2]
)

# Check results
print(df[["Location", "Region"]].head())

                                       Location      Region
0  AFR::CMR::Far North::Goulfey Health District   Far North
1  AFR::CMR::North-West::Batibo Health District  North-West
2         AFR::CMR::East::Kette Health District        East
3     AFR::CMR::Far North::Mada Health District   Far North
4        AFR::CMR::South::Ambam Health District       South


In [36]:
df.to_csv("district_non_phantom_dataset_with_region.csv", index=False)

In [38]:
import pandas as pd

# Load dataset
df = pd.read_csv("district_non_phantom_dataset_with_region.csv")

# Select required columns
selected_df = df[
    [
        "Location",
        "Region",
        "TL",
        "TR",
        "reporting_date",
        "sCh",
        "cCh",
        "deaths",
        "CFR"
    ]
]

# Save to new CSV
selected_df.to_csv(
    "FINAL non_phantom district_dataset_clean.csv",
    index=False
)

print("Dataset saved successfully.")
print(selected_df.head())
print("Shape:", selected_df.shape)

Dataset saved successfully.
                                       Location      Region          TL  \
0  AFR::CMR::Far North::Goulfey Health District   Far North  2017-01-09   
1  AFR::CMR::North-West::Batibo Health District  North-West  2014-06-09   
2         AFR::CMR::East::Kette Health District        East  2010-03-08   
3     AFR::CMR::Far North::Mada Health District   Far North  2015-12-14   
4        AFR::CMR::South::Ambam Health District       South  2016-01-11   

           TR reporting_date  sCh  cCh  deaths  CFR  
0  2017-01-15     2017-01-15   42   17       0  0.0  
1  2014-06-15     2014-06-15    0    0       0  NaN  
2  2010-03-14     2010-03-14    0    0       0  NaN  
3  2015-12-20     2015-12-20    0    0       0  NaN  
4  2016-01-17     2016-01-17    0    0       0  NaN  
Shape: (86339, 9)


In [40]:
import pandas as pd

df = pd.read_csv("Master_non_phantom_Dataset.csv")

# Count region-level records (2 separators)
region_df = df[df["location_parts"] == 2]

print("Region-level records:", len(region_df))

Region-level records: 1142


In [41]:
region_df.to_csv("regional_non_phantom_data_1.csv")

In [42]:
# Load district-level data
df = pd.read_csv("district_non_phantom_dataset_with_region.csv")

# Group districts into regions by reporting date
regional_from_district = (
    df.groupby(["Region","TL","TR", "reporting_date"], as_index=False)
      .agg({
          "sCh": "sum",
          "cCh": "sum",
          "deaths": "sum"
      })
)

In [43]:
# Recalculate CFR
regional_from_district["CFR"] = (
    regional_from_district["deaths"] / regional_from_district["cCh"]
) * 100

regional_from_district.loc[
    regional_from_district["cCh"] == 0, "CFR"
] = 0

# Save
regional_from_district.to_csv("regional_non_phantom_data_2.csv")
print("Rows:", len(regional_from_district))
print("Regions:", regional_from_district["Region"].nunique())

Rows: 4948
Regions: 10


In [44]:
df = pd.read_csv("regional_non_phantom_data_1.csv")

df["Region"] = (
    df["Location"]
    .astype(str)
    .str.split("::")
    .str[-1]
)

In [45]:
df.to_csv("regional_non_phantom_data_1.csv")

In [48]:
regional_data_1 = pd.read_csv("regional_non_phantom_data_1.csv")
regional_data_2 = pd.read_csv("regional_non_phantom_data_2.csv")
print("Regional Data 1:", len(regional_data_1))
print("Regional Data 2:", len(regional_data_2))

Regional Data 1: 1142
Regional Data 2: 4948


In [49]:
# Stack both datasets
combined = pd.concat(
    [regional_data_1, regional_data_2],
    ignore_index=True
)

# Remove duplicates
combined = combined.drop_duplicates(
    subset=["Region", "reporting_date"],
    keep="first"
)

print("Final rows:", len(combined))

# Save
combined.to_csv(
    "regional_non_phantom_master_dataset.csv",
    index=False
)

Final rows: 5295


In [50]:
import pandas as pd

df = pd.read_csv("regional_non_phantom_master_dataset.csv")

# Keep only needed columns
df = df[
    [
        "Region",
        "reporting_date",
        "TL",
        "TR",
        "sCh",
        "cCh",
        "deaths",
        "CFR"
    ]
]

# Save
df.to_csv(
    "regional_non_phantom_master_dataset_clean.csv",
    index=False
)

print(df.head())
print(df.shape)

      Region reporting_date          TL          TR  sCh  cCh  deaths  CFR
0  Far North     2018-11-02  2018-10-31  2018-11-02   91   42       0  0.0
1      North     2018-11-02  2018-10-31  2018-11-02   38   14       0  0.0
2     Centre     2018-11-02  2018-08-27  2018-11-02    0    0       0  NaN
3   Littoral     2018-11-02  2018-10-11  2018-11-02    0    0       0  NaN
4   Adamaoua     2010-12-31  2010-05-06  2010-12-31    1    1       0  0.0
(5295, 8)


In [51]:
import pandas as pd
import requests
import time

df = pd.read_csv("regional_non_phantom_master_dataset_clean.csv")

# Convert dates
df["TL"] = pd.to_datetime(df["TL"])
df["TR"] = pd.to_datetime(df["TR"])

# Approximate regional coordinates for Cameroon
region_coords = {
    "Adamaoua": (7.32, 13.58),
    "Centre": (4.75, 11.83),
    "East": (4.98, 14.30),
    "Far North": (10.59, 14.32),
    "Littoral": (4.05, 9.70),
    "North": (8.65, 13.90),
    "North-West": (6.33, 10.40),
    "South": (2.93, 11.15),
    "South-West": (4.60, 9.30),
    "West": (5.50, 10.50)
}

def fetch_nasa_average(region, start_date, end_date):
    try:
        lat, lon = region_coords[region]

        start = pd.to_datetime(start_date).strftime("%Y%m%d")
        end = pd.to_datetime(end_date).strftime("%Y%m%d")

        url = "https://power.larc.nasa.gov/api/temporal/daily/point"

        params = {
            "parameters": "PRECTOTCORR,T2M,RH2M",
            "community": "AG",
            "longitude": lon,
            "latitude": lat,
            "start": start,
            "end": end,
            "format": "JSON"
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()["properties"]["parameter"]

        rainfall = pd.Series(data["PRECTOTCORR"]).astype(float).mean()
        temperature = pd.Series(data["T2M"]).astype(float).mean()
        humidity = pd.Series(data["RH2M"]).astype(float).mean()

        return rainfall, temperature, humidity

    except Exception as e:
        print(f"Failed for {region}, {start_date} to {end_date}: {e}")
        return None, None, None

In [53]:
# Avoid repeated API calls by fetching each Region + TL + TR once
unique_periods = df[["Region", "TL", "TR"]].drop_duplicates()

results = []

for _, row in unique_periods.iterrows():
    region = row["Region"]

    if region not in region_coords:
        continue

    rainfall, temperature, humidity = fetch_nasa_average(
        region,
        row["TL"],
        row["TR"]
    )

    results.append({
        "Region": region,
        "TL": row["TL"],
        "TR": row["TR"],
        "rainfall_avg": rainfall,
        "temperature_avg": temperature,
        "humidity_avg": humidity
    })

    time.sleep(0.5)  # avoid overwhelming API

env_df = pd.DataFrame(results)

# Merge environmental data into cholera dataset
final_df = df.merge(
    env_df,
    on=["Region", "TL", "TR"],
    how="left"
)

final_df.to_csv("regional_non_phantom_cholera_environment_dataset.csv", index=False)

print("Done.")
print(final_df.head())

Failed for South-West, 2022-04-18 00:00:00 to 2022-04-24 00:00:00: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/daily/point?parameters=PRECTOTCORR%2CT2M%2CRH2M&community=AG&longitude=9.3&latitude=4.6&start=20220418&end=20220424&format=JSON (Caused by NameResolutionError("HTTPSConnection(host='power.larc.nasa.gov', port=443): Failed to resolve 'power.larc.nasa.gov' ([Errno 11001] getaddrinfo failed)"))
Failed for East, 2015-10-19 00:00:00 to 2015-10-25 00:00:00: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/daily/point?parameters=PRECTOTCORR%2CT2M%2CRH2M&community=AG&longitude=14.3&latitude=4.98&start=20151019&end=20151025&format=JSON (Caused by NameResolutionError("HTTPSConnection(host='power.larc.nasa.gov', port=443): Failed to resolve 'power.larc.nasa.gov' ([Errno 11001] getaddrinfo failed)"))
Failed for Littoral, 2010-10-18 00:00:00 to 2010-10-24 00:00:00: HTTPSConnec

In [55]:
df=pd.read_csv("regional_non_phantom_cholera_environment_dataset.csv")

In [56]:
cols = [
    "cCh",
    "sCh",
    "deaths",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]

print(df[cols].corr())

                      cCh       sCh    deaths  rainfall_avg  temperature_avg  \
cCh              1.000000  0.908191  0.845501      0.163200         0.059275   
sCh              0.908191  1.000000  0.785339      0.165661         0.079141   
deaths           0.845501  0.785339  1.000000      0.090950         0.055304   
rainfall_avg     0.163200  0.165661  0.090950      1.000000        -0.260428   
temperature_avg  0.059275  0.079141  0.055304     -0.260428         1.000000   
humidity_avg     0.100946  0.107046  0.053941      0.508520        -0.485024   

                 humidity_avg  
cCh                  0.100946  
sCh                  0.107046  
deaths               0.053941  
rainfall_avg         0.508520  
temperature_avg     -0.485024  
humidity_avg         1.000000  


In [57]:
# 2. Convert date
df["reporting_date"] = pd.to_datetime(df["reporting_date"], errors="coerce")

# 3. Convert case columns to numeric
df["sCh"] = pd.to_numeric(df["sCh"], errors="coerce").fillna(0)
df["cCh"] = pd.to_numeric(df["cCh"], errors="coerce").fillna(0)

# 4. Create total cases
df["cases"] = df["sCh"] + df["cCh"]

# 5. Create month column
df["month_date"] = df["reporting_date"].dt.to_period("M").dt.to_timestamp()

# 6. Aggregate to monthly regional level
monthly = (
    df.groupby(["Region", "month_date"])
    .agg({
        "cases": "sum",
        "rainfall_avg": "mean",
        "temperature_avg": "mean",
        "humidity_avg": "mean"
    })
    .reset_index()
)

# 7. Sort data
monthly = monthly.sort_values(["Region", "month_date"])

# 8. Check result
print(monthly.head(20))
print(monthly.shape)
print(monthly["cases"].describe())
print(monthly["Region"].nunique())

                                               Region month_date  cases  \
0                   (Centre|Far North|Littoral|North) 2018-11-01   1465   
1                   (Centre|Far North|Littoral|North) 2018-12-01   2567   
2   (Centre|Far North|Littoral|North|South-West|West) 2010-10-01  11646   
3                             (Centre|Littoral|North) 2018-09-01    584   
4                             (Centre|Littoral|North) 2018-10-01    515   
5                  (Centre|Littoral|South|South-West) 2020-12-01   2491   
6                                      (Centre|North) 2018-08-01    619   
7               (Far North|Littoral|North|South-West) 2020-01-01   2802   
8                                   (Far North|North) 2018-11-01    400   
9                        (Far North|North|South-West) 2020-01-01   2972   
10                                   (Littoral|North) 2018-10-01     55   
11                                           Adamaoua 2010-01-01     19   
12                       

In [58]:
monthly.groupby("Region").size().sort_values()

Region
(Centre|Far North|Littoral|North|South-West|West)      1
(Centre|Littoral|South|South-West)                     1
(Far North|Littoral|North|South-West)                  1
(Centre|North)                                         1
(Far North|North)                                      1
(Far North|North|South-West)                           1
(Littoral|North)                                       1
(Centre|Littoral|North)                                2
(Centre|Far North|Littoral|North)                      2
Adamaoua                                             113
East                                                 113
North-West                                           113
West                                                 121
North                                                127
Far North                                            127
Littoral                                             129
South-West                                           129
South                   

In [59]:
print("Total rows:", len(monthly))

print("Zero case rows:",
      (monthly["cases"] == 0).sum())

print("Percent zeros:",
      round(
          (monthly["cases"] == 0).mean() * 100,
          2
      ),
      "%"
)

Total rows: 1248
Zero case rows: 114
Percent zeros: 9.13 %


In [60]:
monthly = monthly.sort_values(
    ["Region", "month_date"]
)

monthly["cases_lag1"] = (
    monthly.groupby("Region")["cases"]
    .shift(1)
)

monthly["cases_lag2"] = (
    monthly.groupby("Region")["cases"]
    .shift(2)
)

monthly["cases_lag3"] = (
    monthly.groupby("Region")["cases"]
    .shift(3)
)

In [61]:
monthly["future_cases"] = (
    monthly.groupby("Region")["cases"]
    .shift(-1)
)

In [62]:
monthly = monthly.dropna()

In [63]:
print(
    monthly[
        [
            "Region",
            "month_date",
            "cases",
            "cases_lag1",
            "cases_lag2",
            "cases_lag3",
            "future_cases"
        ]
    ].head(20)
)

      Region month_date  cases  cases_lag1  cases_lag2  cases_lag3  \
14  Adamaoua 2010-04-01    378       154.0        47.0        19.0   
15  Adamaoua 2010-05-01     86       378.0       154.0        47.0   
16  Adamaoua 2010-06-01    126        86.0       378.0       154.0   
17  Adamaoua 2010-07-01    479       126.0        86.0       378.0   
18  Adamaoua 2010-08-01    204       479.0       126.0        86.0   
19  Adamaoua 2010-09-01    120       204.0       479.0       126.0   
20  Adamaoua 2010-10-01     62       120.0       204.0       479.0   
21  Adamaoua 2010-11-01    128        62.0       120.0       204.0   
22  Adamaoua 2010-12-01     82       128.0        62.0       120.0   
23  Adamaoua 2011-01-01    161        82.0       128.0        62.0   
24  Adamaoua 2011-02-01      0       161.0        82.0       128.0   
25  Adamaoua 2011-03-01     75         0.0       161.0        82.0   
26  Adamaoua 2011-04-01    209        75.0         0.0       161.0   
27  Adamaoua 2011-05

In [64]:
print(
    monthly[
        [
            "cases",
            "cases_lag1",
            "cases_lag2",
            "cases_lag3",
            "future_cases"
        ]
    ].corr()
)

                 cases  cases_lag1  cases_lag2  cases_lag3  future_cases
cases         1.000000    0.634679    0.498327    0.331475      0.629956
cases_lag1    0.634679    1.000000    0.682117    0.535870      0.487833
cases_lag2    0.498327    0.682117    1.000000    0.681611      0.324003
cases_lag3    0.331475    0.535870    0.681611    1.000000      0.199794
future_cases  0.629956    0.487833    0.324003    0.199794      1.000000


In [65]:
import numpy as np

monthly["future_cases_log"] = np.log1p(
    monthly["future_cases"]
)

print(
    monthly["future_cases_log"].describe()
)

count    1196.000000
mean        5.490134
std         2.129345
min         0.000000
25%         4.875197
50%         6.042607
75%         6.826002
max         9.627602
Name: future_cases_log, dtype: float64


In [66]:
print(monthly["future_cases"].describe())
print(monthly["future_cases_log"].describe())

count     1196.000000
mean       744.329431
std       1143.872049
min          0.000000
25%        130.000000
50%        420.000000
75%        920.500000
max      15177.000000
Name: future_cases, dtype: float64
count    1196.000000
mean        5.490134
std         2.129345
min         0.000000
25%         4.875197
50%         6.042607
75%         6.826002
max         9.627602
Name: future_cases_log, dtype: float64


In [67]:
features = [
    "cases",
    "cases_lag1",
    "cases_lag2",
    "cases_lag3",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg",
    "month",
    "Region_enc"
]

In [68]:
import numpy as np

monthly["future_cases_log"] = np.log1p(
    monthly["future_cases"]
)

y = monthly["future_cases_log"]

In [69]:
print(monthly.shape)
print(df.shape)

(1196, 11)
(5295, 13)


In [70]:
train = monthly[
    monthly["month_date"] < "2021-01-01"
]

test = monthly[
    monthly["month_date"] >= "2021-01-01"
]

print(train.shape)
print(test.shape)

print(train["month_date"].min())
print(train["month_date"].max())

print(test["month_date"].min())
print(test["month_date"].max())

(1145, 11)
(51, 11)
2010-04-01 00:00:00
2020-12-01 00:00:00
2021-10-01 00:00:00
2022-12-01 00:00:00


In [71]:
monthly[
    (monthly["month_date"] >= "2021-01-01") &
    (monthly["month_date"] < "2021-10-01")
].shape

(0, 11)

In [72]:
monthly[
    (monthly["month_date"] >= "2021-01-01") &
    (monthly["month_date"] < "2021-10-01")
][["Region","month_date"]].head(20)

,Region,month_date


In [73]:
features = [
    "cases",
    "cases_lag1",
    "cases_lag2",
    "cases_lag3",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]

print(
    monthly[
        features + ["future_cases"]
    ].corr()["future_cases"]
    .sort_values(ascending=False)
)

future_cases       1.000000
cases              0.629956
cases_lag1         0.487833
cases_lag2         0.324003
rainfall_avg       0.239651
cases_lag3         0.199794
temperature_avg    0.152145
humidity_avg       0.120182
Name: future_cases, dtype: float64


In [74]:
q1 = monthly["future_cases"].quantile(0.33)
q2 = monthly["future_cases"].quantile(0.66)

print(q1, q2)

211.0 698.7


In [75]:
def risk_label(x):

    if x <= q1:
        return 0  

    elif x <= q2:
        return 1  

    return 2      

monthly["future_risk"] = (
    monthly["future_cases"]
    .apply(risk_label)
)

In [79]:
from xgboost import XGBClassifier

In [81]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(df['Region'])

region_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print(region_mapping)

{'(Centre|Far North|Littoral|North)': np.int64(0), '(Centre|Far North|Littoral|North|South-West|West)': np.int64(1), '(Centre|Littoral|North)': np.int64(2), '(Centre|Littoral|South|South-West)': np.int64(3), '(Centre|North)': np.int64(4), '(Far North|Littoral|North|South-West)': np.int64(5), '(Far North|North)': np.int64(6), '(Far North|North|South-West)': np.int64(7), '(Littoral|North)': np.int64(8), 'Adamaoua': np.int64(9), 'Centre': np.int64(10), 'East': np.int64(11), 'Far North': np.int64(12), 'Littoral': np.int64(13), 'North': np.int64(14), 'North-West': np.int64(15), 'South': np.int64(16), 'South-West': np.int64(17), 'West': np.int64(18)}


In [82]:
region_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print(region_mapping)

{'(Centre|Far North|Littoral|North)': np.int64(0), '(Centre|Far North|Littoral|North|South-West|West)': np.int64(1), '(Centre|Littoral|North)': np.int64(2), '(Centre|Littoral|South|South-West)': np.int64(3), '(Centre|North)': np.int64(4), '(Far North|Littoral|North|South-West)': np.int64(5), '(Far North|North)': np.int64(6), '(Far North|North|South-West)': np.int64(7), '(Littoral|North)': np.int64(8), 'Adamaoua': np.int64(9), 'Centre': np.int64(10), 'East': np.int64(11), 'Far North': np.int64(12), 'Littoral': np.int64(13), 'North': np.int64(14), 'North-West': np.int64(15), 'South': np.int64(16), 'South-West': np.int64(17), 'West': np.int64(18)}


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd


df = pd.read_csv("FINAL regional_cholera_environment_dataset.csv")

monthly["month_date"] = df["TR"].dt.to_period("M").dt.to_timestamp()

monthly["year"] = monthly["month_date"].dt.year
monthly["month"] = monthly["month_date"].dt.month

# Cyclical month encoding
monthly["month_sin"] = np.sin(2*np.pi * monthly["month"]/12)
monthly["month_cos"] = np.cos(2*np.pi * monthly["month"]/12)
# Baseline features
baseline_features = [
    "cases",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg",
    "month",
    "Region",
    "deaths"
]

X_base = monthly[baseline_features]
y = monthly["risk_level"]

# One-hot encode region
X_base = pd.get_dummies(
    X_base,
    columns=["Region"],
    prefix="Region_"
)

# Logistic Regression model with scaling
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=10000,
        solver="lbfgs"
    )
)

# Time series cross validation
tscv = TimeSeriesSplit(n_splits=5)

summary = []

accuracies = []
f1_macros = []

precisions = {
    "Low": [],
    "Medium": [],
    "High": []
}

recalls = {
    "Low": [],
    "Medium": [],
    "High": []
}


# Cross-validation loop
for train_idx, test_idx in tscv.split(X_base):

    X_train = X_base.iloc[train_idx]
    X_test = X_base.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        target_names=["Low", "Medium", "High"],
        output_dict=True,
        zero_division=0
    )

    # Macro F1-score
    f1_macros.append(
        report["macro avg"]["f1-score"]
    )

    # Class-specific precision and recall
    for cls in ["Low", "Medium", "High"]:
        precisions[cls].append(
            report[cls]["precision"]
        )

        recalls[cls].append(
            report[cls]["recall"]
        )


# Store averaged results
summary.append({

    "Model": "Logistic Regression",

    "Accuracy": float(np.mean(accuracies)),

    "Macro F1": float(np.mean(f1_macros)),

    "Low Precision": float(np.mean(precisions["Low"])),
    "Low Recall": float(np.mean(recalls["Low"])),

    "Medium Precision": float(np.mean(precisions["Medium"])),
    "Medium Recall": float(np.mean(recalls["Medium"])),

    "High Precision": float(np.mean(precisions["High"])),
    "High Recall": float(np.mean(recalls["High"]))

})


# Convert results into dataframe
df_summary = pd.DataFrame(summary)

# Display results
print(df_summary.round(4))

AttributeError: Can only use .dt accessor with datetimelike values